## Mapping NEVO to FoodOn using reranking by mapping back

## **Import packages**

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from openai import AzureOpenAI, OpenAI

## **Load FoodOn, NEVO, FoodData Central and KAP data**

### FoodOn

In [ ]:
## Read in FoodOn ontology as a table
if os.path.exists('../data/intermediate/df_foodon_processed.pkl'):
    foodon_path = '../data/intermediate/df_foodon_processed.pkl'
    df_foodon = pd.read_pickle(foodon_path)
else:
    foodon_path = '../data/input/FOODON.csv'
    df_foodon = pd.read_csv(foodon_path)

In [ ]:
df_foodon

### NEVO

In [ ]:
## Read in NEVO data
if os.path.exists('../data/intermediate/df_nevo_processed.pkl'):
    nevo_path = '../data/intermediate/df_nevo_processed.pkl'
    df_nevo = pd.read_pickle(nevo_path)
else:
    nevo_path = '../data/input/NEVO2023_v8.0.xlsx'
    df_nevo = pd.read_excel(nevo_path)


### FoodData Central
1. FoodData Central Foundation food - food
2. FoodData Central Foundation food - food category
3. FoodData Central SR Legacy - food
4. FoodData Central SR Legacy - food category

In [ ]:
## Read in FoodData Central
if os.path.exists('../data/intermediate/df_fooddatacentral_processed.pkl'):
    fooddatacentral_path = '../data/intermediate/df_fooddatacentral_processed.pkl'
    df_fooddatacentral = pd.read_pickle(fooddatacentral_path)
else:
    # 1. FoodData Central Foundation food - food
    fooddatacentral_foundation_food = pd.read_csv("../data/input/FoodData_Central_foundation_food.csv")
    # 2. FoodData Central Foundation food - food category
    fooddatacentral_foundation_food_category = pd.read_csv("../data/input/FoodData_Central_foundation_food_category.csv")
    # 3. FoodData Central SR Legacy - food
    fooddatacentral_legacy_food = pd.read_csv("../data/input/FoodData_Central_legacy_food.csv")
    # 4. FoodData Central SR Legacy - food category
    fooddatacentral_legacy_food_category = pd.read_csv("../data/input/FoodData_Central_legacy_food_category.csv")

    # Filter on data type 'foundation_food' for foundation food and 'sr_legacy_food' for SR legacy
    fooddatacentral_foundation_food = fooddatacentral_foundation_food[fooddatacentral_foundation_food['data_type'] == 'foundation_food']
    fooddatacentral_foundation_food = fooddatacentral_foundation_food.drop_duplicates(subset=['description'], keep='last')
    fooddatacentral_legacy_food = fooddatacentral_legacy_food[fooddatacentral_legacy_food['data_type'] == 'sr_legacy_food']
    # Left join foundation food and foundation food category
    fooddatacentral_foundation = pd.merge(fooddatacentral_foundation_food, fooddatacentral_foundation_food_category, how="left", left_on="food_category_id", right_on="id")
    # Left join sr legacy food and sr legacy food category
    fooddatacentral_legacy = pd.merge(fooddatacentral_legacy_food, fooddatacentral_legacy_food_category, how="left", left_on="food_category_id", right_on="id")

### KAP

In [ ]:
## Read in KAP
kap_path = "../data/input/df_kap.json"
df_kap = pd.read_json(kap_path, lines=True)

## **Preprocessing and creating embeddings**

In [ ]:
from helpers.step1 import Embedder, get_all_children_ids, get_parent_ids

## FoodOn

In [ ]:
# Only if not done before
if not os.path.exists('../data/intermediate/df_foodon_processed.pkl'):
    # We take the FoodOn ontology version 2025-02-01. Specifically, we use the ‘food product’ subset of FoodOn, 
    # containing the ‘food product’ entity and all its child nodes.
    # In the FoodOn extraction, a string for each food was created by joining all synonyms and parent labels (between brackets) 
    # for each food entity, except for the ‘food product’ entity, because it has no parent due to the subset we use. For example, 
    # for the food product ‘pork roast (boneless, raw)’ with synonym ‘pig roast (boneless, raw)’ and parents ‘pork roast (raw)’ 
    # and ‘pork roast (boneless)’, after preprocessing we got ‘pork roast (boneless, raw) pig roast (boneless, raw) (pork roast 
    # (raw), pork roast (boneless))’.
    
    foodon_food_names = []
    
    # Filter all 'food products' (Class ID = http://purl.obolibrary.org/obo/FOODON_00001002), i.e. children of item 'food product' including 'food product' itself
    all_food_product_ids = get_all_children_ids(df_foodon, "http://purl.obolibrary.org/obo/FOODON_00001002", all_ids = [])
    
    df_foodon_food_products = df_foodon[df_foodon['Class ID'].isin(all_food_product_ids)].reset_index()
    df_foodon_food_products.shape

    # Preprocess all food products
    for i in range(0, df_foodon_food_products.shape[0]):
        # First, set label as food name
        orig_food_name = df_foodon_food_products['label'].iloc[i]
        foodon_food_name = orig_food_name

        # Then, add synonyms, if they exist
        synonyms = df_foodon_food_products['Synonyms'].iloc[i] 
        if synonyms is not np.nan:
            for synonym in synonyms.split("|"):
                if foodon_food_name != foodon_food_name: # e.g. foodon_food_name is nan
                    foodon_food_name = synonym
                else:
                    foodon_food_name = foodon_food_name + ", " + synonym

        # Finally, add parent object (if it exists) between brackets
        parent_ids = get_parent_ids(df_foodon=df_foodon_food_products, foodon_id=df_foodon_food_products['Class ID'].iloc[i])
        parent_food_row = df_foodon_food_products[df_foodon_food_products['Class ID'].isin(parent_ids)]
        if len(parent_food_row) > 0:
            parent_food_row = parent_food_row.reset_index()
            # Add first parent object
            foodon_food_name += " (" + parent_food_row["label"][0]
            
            # If there are more, add them separated by comma
            for row in range(1, len(parent_food_row)): # Start from index 1, since index 0 has already been added
                foodon_food_name += ", " + parent_food_row["label"][row]
            
            # Finally, close with closing bracket
            foodon_food_name += ")"
        
        # Append to foodon_food_names
        foodon_food_names.append(foodon_food_name)

    # For all food items we generate the embeddings with the text embedding model 'text-embedding-ada-002' by OpenAI.
    embedder = Embedder()
    foodon_food_embeddings = []
    for foodon_food_name in foodon_food_names:
        foodon_embedding = embedder.create_embedding(foodon_food_name)
        foodon_food_embeddings.append(foodon_embedding)

    df_foodon_processed = pd.DataFrame({"id": df_foodon_food_products['Class ID'], "name": foodon_food_names, "embedding": foodon_food_embeddings})

else:
    # otherwise, use already saved dataframe
    df_foodon_processed = df_foodon
df_foodon_processed.head()

## NEVO

In [ ]:
# Only if not done before
if not os.path.exists('../data/intermediate/df_nevo_processed.pkl'):
    # We take from NEVO version 2023 8.0 the columns ‘Engelse naam/Food name’ with the English names of food items,
    # combined with the column ‘Food Group’, containing the food group in English, which is added between brackets. 
    # E.g.: ‘Tamarind (Herbs and Spices)’ and ‘Radish raw (Vegetables)’. 
    nevo_food_names = list(df_nevo['Engelse naam/Food name'] + " (" + df_nevo["Food group"] + ")")

    # For all food items in NEVO we generate vector embeddings with 'text-embedding-ada-002'
    embedder = Embedder()
    nevo_food_embeddings = []
    for nevo_food_name in nevo_food_names:
        nevo_embedding = embedder.create_embedding(nevo_food_name)
        nevo_food_embeddings.append(nevo_embedding)

    # Put names and embeddings together in pandas dataframe
    df_nevo_processed = pd.DataFrame({"id": df_nevo['NEVO-code'], "name": nevo_food_names, "embedding": nevo_food_embeddings})
else:
    # otherwise, use already saved dataframe
    df_nevo_processed = df_nevo
df_nevo_processed.head()

## FoodData Central

In [ ]:
# Only if not done before
if not os.path.exists('../data/intermediate/df_fooddatacentral_processed.pkl'):
    # Preprocess FoodData Central data
    df_fooddatacentral = pd.concat([fooddatacentral_foundation, fooddatacentral_legacy])
    df_fooddatacentral['name'] = df_fooddatacentral['description_x'] + " (" + df_fooddatacentral['description_y'] + ")"
    df_fooddatacentral = df_fooddatacentral.drop_duplicates(subset=['name']).reset_index() # Remove duplicates, both in SR Legacy and Foundation Foods
    df_fooddatacentral = df_fooddatacentral[['fdc_id', 'name']]

    # Get food names from FoodData Central
    fooddatacentral_food_names = list(df_fooddatacentral['name'])

    # For all food items in FoodData Central we generate vector embeddings with 'text-embedding-ada-002'
    embedder = Embedder()
    fooddatacentral_food_embeddings = []
    for fooddatacentral_food_name in fooddatacentral_food_names:
        fooddatacentral_embedding = embedder.create_embedding(fooddatacentral_food_name)
        fooddatacentral_food_embeddings.append(fooddatacentral_embedding)

    # Put names and embeddings together in pandas dataframe
    df_fooddatacentral_processed = pd.DataFrame({"id": df_fooddatacentral['fdc_id'], "name": fooddatacentral_food_names, "embedding": fooddatacentral_food_embeddings})

else:
    # otherwise, use already saved dataframe
    df_fooddatacentral_processed = df_fooddatacentral
    
df_fooddatacentral_processed.head()

### Save intermediate embedding results

In [ ]:
if not os.path.exists('../data/intermediate/df_nevo_processed.pkl'):
    # Save intermediate NEVO
    df_nevo_processed.to_pickle('../data/intermediate/df_nevo_processed.pkl')

if not os.path.exists('../data/intermediate/df_foodon_processed.pkl'):
    # Save intermediate FoodOn
    df_foodon_processed.to_pickle('../data/intermediate/df_foodon_processed.pkl')

if not os.path.exists('../data/intermediate/df_fooddatacentral_processed.pkl'):
    df_fooddatacentral_processed.to_pickle('../data/intermediate/df_fooddatacentral_processed.pkl')